### Preprocessing final results

In [1]:
results_path = "globecom_approaches_results_with_scores.json"

In [2]:
import pandas as pd
df = pd.read_json(results_path)

In [3]:
for col in df.columns:
    for idx in df.index:
        df.at[idx, col].pop("BoT_creation_cost")
        df.at[idx, col].pop("prompt_generation_cost")
        df.at[idx, col].pop("scores")

In [4]:
df.to_json("globecom_final_results.json", indent=4)

### Getting generations

In [7]:
final_results_path = "globecom_final_results.json"

In [8]:
import json
with open(final_results_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [9]:
pca_generations = {}
ica_generations = {}
swpca_generations = {}
gw_generations = {}

methods_map = {
    "pca": pca_generations,
    "ica": ica_generations,
    "score_weighted_pca": swpca_generations,
    "gradient_weighted": gw_generations,
}


In [10]:
for method_name, dest_dict in methods_map.items():
    method_content = data[method_name]
    for idx, result_dict in method_content.items():
        malicious_req = result_dict["malicious_request"]
        answers = {}
        for target_model, target_model_results in result_dict['targets'].items():
            answers[target_model] = [a for a in target_model_results['target_responses']]
        dest_dict[malicious_req] = answers
    

In [11]:
with open("generations/pca_generations.json", "w") as f:
    json.dump(pca_generations, f, indent=4)
with open("generations/ica_generations.json", "w") as f:
    json.dump(ica_generations, f, indent=4)
with open("generations/swpca_generations.json", "w") as f:
    json.dump(swpca_generations, f, indent=4)
with open("generations/gw_generations.json", "w") as f:
    json.dump(gw_generations, f, indent=4)

In [12]:
len(gw_generations.keys())

100

### Preprocessing AUTODAN generations

In [13]:
with open("Llama-3.1-8B-Instruct/harmbench/meta-llama-Llama-3.1-8B-Instruct-generations.json", "r", encoding="utf-8") as f:
    autodan_generations = json.load(f)

In [14]:
new_generations = {}
for key, value in autodan_generations.items():
    mal_req = value["malicious_request"]
    if mal_req in gw_generations.keys():
        new_generations[mal_req] = [item["target_response"] for item in value]


TypeError: list indices must be integers or slices, not str

In [22]:
len(new_generations.keys())

100

In [23]:
with open("generations/autodan/autodan_generations.json", "w", encoding="utf-8") as f:
    json.dump(new_generations, f, indent=4)